# Scientific Paper Markdown — Stage 0 Parser & Debug Lab

**Input:** `EDLECOC(1).md` (the supplied example paper)  
**Output:** `ParsedPaper.json` + `SectionCatalog.json`, with source-linked sections/chunks/blocks, scientific elements, search, validation, and debug reports.

This notebook implements **deterministic preprocessing only**: no LLM calls, no paper summarization, no external browsing, and no automatic modification of the OCR source. All source locations are **1-based, inclusive** original Markdown line numbers. Each major function is implemented and exercised in a separate cell, so you can stop, edit, and re-run individual stages.

> **Dependency:** `markdown-it-py` (`pip install markdown-it-py`). Everything else uses Python's standard library; `pandas` and IPython improve inspection but are optional.

**Important known limitations:** Author extraction is a heuristic; GFM pipe-table handling requires correct Markdown table syntax; semantic citation attribution is **not** part of preprocessing; display-math parsing handles common delimiters but is not a full TeX parser; token counts are approximate; evidence validation later must still check meaning, not just source location.

## 0 · Configuration and imports

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
from hashlib import sha256
from bisect import bisect_right
from itertools import chain
from pprint import pprint
import json
import re

try:
    from markdown_it import MarkdownIt
except ImportError as exc:
    raise ImportError('Install markdown-it-py: %pip install markdown-it-py') from exc

try:
    import pandas as pd
    from IPython.display import display
except ImportError:
    pd = None
    def display(x): print(x)

# Put the uploaded Markdown beside this notebook, or change this path manually.
SOURCE_CANDIDATES = [Path('FIGS-BD.md')]  # edit for your manuscript
SOURCE_PATH = next((p for p in SOURCE_CANDIDATES if p.is_file()), None)
assert SOURCE_PATH, 'Input not found: edit SOURCE_CANDIDATES.'
RUN_EDLECOC_REGRESSION = False  # True only for the exact example Markdown

OUTPUT_DIR = SOURCE_PATH.parent / 'paper_parser_outputs'
MAX_CHUNK_TOKENS = 1100  # estimated tokens; tune here
PARSER_VERSION = 'stage0-debug-1.0'

def show_rows(rows, n=12):
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]: pprint(row)
    if len(rows) > n: print(f'... {len(rows)-n} more rows')

print('Source:', SOURCE_PATH.resolve())
print('Parser:', PARSER_VERSION)

Source: /Users/emretuygan/Desktop/Research_agent_shared/VaultManager/research_agent_v4 2/FIGS-BD.md
Parser: stage0-debug-1.0


## 1 · Load source, preserve exact lines, calculate immutable fingerprint

In [3]:
def load_source(path: Path) -> dict:
    # newline='' prevents universal-newline conversion, preserving source bytes.
    with path.open('r', encoding='utf-8-sig', newline='') as handle:
        source = handle.read()
    lines = source.splitlines(keepends=True)
    starts = [0]
    for line in lines:
        starts.append(starts[-1] + len(line))
    return {
        'path': str(path.resolve()), 'text': source,
        'lines': lines, 'line_starts': starts,
        'sha256': sha256(path.read_bytes()).hexdigest(),
        'line_count': len(lines)
    }

source = load_source(SOURCE_PATH)

def source_lines(start_line, end_line):
    """Exact source slice; start/end are 1-based INCLUSIVE."""
    assert 1 <= start_line <= end_line <= source['line_count']
    return ''.join(source['lines'][start_line-1:end_line])

def char_to_line(char_pos):
    return bisect_right(source['line_starts'], char_pos)  # 1-based

print(f"Lines: {source['line_count']} | Characters: {len(source['text']):,} | SHA-256: {source['sha256'][:20]}...")
print('First three lines:\n' + source_lines(1, min(3, source['line_count'])))

Lines: 124 | Characters: 19,447 | SHA-256: b980864d727aac686fbc...
First three lines:
# Adaptive Test-Time Intervention for Concept Bottleneck Models

**Matthew Shen\*, Aliyah R. Hsu\*, Abhineet Agarwal, and Bin Yu**  



## 2 · Parse Markdown structurally (no regex-based heading splitting)

In [4]:
# The table extension is important for the ECOC results tables.
markdown_parser = MarkdownIt('commonmark').enable('table')
tokens = markdown_parser.parse(source['text'])

print(f'Parsed {len(tokens)} Markdown tokens.')
print('Root token types:', Counter(t.type for t in tokens).most_common(12))
print('\nFirst heading tokens:')
for t in (t for t in tokens if t.type == 'heading_open'):
    print(f'{t.tag:>2s} | L{t.map[0]+1:3d} | {source_lines(t.map[0]+1, t.map[1]).strip()}')

Parsed 356 Markdown tokens.
Root token types: [('inline', 104), ('td_open', 46), ('td_close', 46), ('paragraph_open', 39), ('paragraph_close', 39), ('tr_open', 13), ('tr_close', 13), ('heading_open', 12), ('heading_close', 12), ('th_open', 7), ('th_close', 7), ('list_item_open', 2)]

First heading tokens:
h1 | L  1 | # Adaptive Test-Time Intervention for Concept Bottleneck Models
h2 | L 14 | ## Abstract
h2 | L 20 | ## 1. Introduction
h2 | L 33 | ## 2. Related Work
h3 | L 35 | ### 2.1 Concept Models
h3 | L 39 | ### 2.2 Knowledge and Model Distillation
h2 | L 43 | ## 3. FIGS Binary Distillation – FIGS-BD
h2 | L 61 | ## 4. Datasets and Teacher Models
h2 | L 67 | ## 5. Distillation and Prediction Performance
h2 | L 84 | ## 6. Adaptive Test-Time Intervention Using FIGS-BD
h2 | L118 | ## 7. Conclusion
h2 | L122 | ## Acknowledgements


## 3 · Build section hierarchy (subtrees, local spans, reading order)

In [5]:
def parse_sections(tokens, lines):
    headings = []
    for token in tokens:
        if token.type != 'heading_open' or token.map is None: continue
        headings.append({
            'title': re.sub(r'^\s{0,3}#{1,6}\s*', '', lines[token.map[0]].rstrip('\r\n')).strip(' #'),
            'level': int(token.tag[1:]),
            'start_line': token.map[0] + 1
        })
    sections = []
    if not headings or headings[0]['start_line'] > 1:
        sections.append({
            'section_id': 'S000', 'title': 'UNSECTIONED', 'level': 0,
            'parent_id': None, 'child_ids': [], 'line_start': 1,
            'local_end': (headings[0]['start_line'] - 1) if headings else len(lines),
            'subtree_end': (headings[0]['start_line'] - 1) if headings else len(lines),
            'kind': 'front_matter', 'heading_path': []
        })
    stack = []
    by_id = {s['section_id']:s for s in sections}
    for i, h in enumerate(headings):
        while stack and stack[-1]['level'] >= h['level']: stack.pop()
        parent = stack[-1] if stack else None
        sid = f'S{i+1:03d}'
        title = h['title']
        normalized = re.sub(r'^[IVXLC]+\.\s*', '', title, flags=re.I).strip()
        if re.match(r'^(references|bibliography|works cited)\s*$', normalized, flags=re.I):
            kind = 'bibliography'
        elif re.match(r'^(appendix|appendices|supplementary|supplemental)\b', normalized, flags=re.I):
            kind = 'appendix'
        elif normalized.lower() == 'abstract': kind = 'abstract'
        else: kind = 'body'
        section = {
            'section_id': sid, 'title': title, 'level': h['level'],
            'parent_id': parent['section_id'] if parent else None,
            'child_ids': [], 'line_start': h['start_line'],
            'local_end': len(lines), 'subtree_end': len(lines),
            'kind': kind,
            'heading_path': ([*parent['heading_path']] if parent else []) + [title]
        }
        if parent: parent['child_ids'].append(sid)
        sections.append(section)
        by_id[sid] = section
        stack.append(section)
    # Every line belongs to exactly one local section (unsectioned or latest heading).
    for i, section in enumerate(sections):
        if section['section_id'] == 'S000': continue
        if i+1 < len(sections):
            section['local_end'] = sections[i+1]['line_start']-1
        for later in sections[i+1:]:
            if later['level'] <= section['level']:
                section['subtree_end'] = later['line_start']-1
                break
    # OCR may place acknowledgements or a publisher footer after the References list,
    # separated only by a horizontal rule. Create a virtual back-matter section rather
    # than mistakenly treating that text as the final bibliography entry.
    tail_bib = next((s for s in reversed(sections)
                     if s['kind']=='bibliography' and s['subtree_end']==len(lines)), None)
    if tail_bib:
        hr_positions = [t.map[0]+1 for t in tokens if t.type=='hr' and t.map
                        and t.map[0]+1 > tail_bib['line_start']]
        boundary = next((n for n in hr_positions
                         if any(line.strip() for line in lines[n:]) and
                         any(re.match(r'^\s*\[\d+\]',line) for line in lines[tail_bib['line_start']:n-1])), None)
        if boundary:
            tail_bib['local_end'] = boundary-1
            tail_bib['subtree_end'] = boundary-1
            parent = by_id.get(tail_bib['parent_id'])
            sid = f'S{len(headings)+1:03d}'
            back = {'section_id':sid, 'title':'BACK_MATTER_AFTER_REFERENCES',
                    'level':tail_bib['level'], 'parent_id':tail_bib['parent_id'],
                    'child_ids':[], 'line_start':boundary, 'local_end':len(lines),
                    'subtree_end':len(lines), 'kind':'back_matter',
                    'heading_path': ([*parent['heading_path']] if parent else [])+['BACK_MATTER_AFTER_REFERENCES']}
            sections.append(back)
            if parent:parent['child_ids'].append(sid)
    return sections

sections = parse_sections(tokens, source['lines'])
section_by_id = {s['section_id']: s for s in sections}

def section_for_line(line):
    # local section owner; deepest heading seen so far
    matching = [s for s in sections if s['line_start'] <= line <= s['local_end']]
    return matching[0] if len(matching) == 1 else None

show_rows([{'id':s['section_id'],'level':s['level'],'title':s['title'],
            'parent':s['parent_id'],'local':f"{s['line_start']}-{s['local_end']}",
            'subtree_end':s['subtree_end'],'kind':s['kind']} for s in sections], n=25)

,id,level,title,parent,local,subtree_end,kind
0,S001,1,Adaptive Test-Time Intervention for Concept Bo...,NaN,1-13,124,body
1,S002,2,Abstract,S001,14-19,19,abstract
2,S003,2,1. Introduction,S001,20-32,32,body
3,S004,2,2. Related Work,S001,33-34,42,body
4,S005,3,2.1 Concept Models,S004,35-38,38,body
5,S006,3,2.2 Knowledge and Model Distillation,S004,39-42,42,body
6,S007,2,3. FIGS Binary Distillation – FIGS-BD,S001,43-60,60,body
7,S008,2,4. Datasets and Teacher Models,S001,61-66,66,body
8,S009,2,5. Distillation and Prediction Performance,S001,67-83,83,body
9,S010,2,6. Adaptive Test-Time Intervention Using FIGS-BD,S001,84-117,117,body


## 4 · Extract top-level source blocks without duplicating nested list/table content

In [6]:
def parse_blocks(tokens, source):
    """Map root Markdown nodes onto exact source spans. Preserve headings as blocks."""
    root_kinds = {
        'heading_open':'heading', 'paragraph_open':'paragraph',
        'bullet_list_open':'list', 'ordered_list_open':'list',
        'table_open':'table', 'blockquote_open':'blockquote',
        'fence':'code', 'code_block':'code', 'html_block':'html',
        'hr':'rule'
    }
    depth = 0
    result = []
    for token in tokens:
        if depth == 0 and token.type in root_kinds and token.map:
            start, end = token.map[0]+1, token.map[1]  # map.end is exclusive (0-based)
            result.append({
                'block_id': f'B{len(result)+1:03d}',
                'kind':root_kinds[token.type], 'line_start':start, 'line_end':end,
                'section_id':section_for_line(start)['section_id'] if section_for_line(start) else None,
                'source_hash':sha256(source_lines(start, end).encode('utf-8')).hexdigest()[:16]
            })
        depth = max(0, depth + token.nesting)
    return result

blocks = parse_blocks(tokens, source)
blocks_by_id = {b['block_id']:b for b in blocks}
print('Block breakdown:', dict(Counter(b['kind'] for b in blocks)))
show_rows(blocks, 16)

Block breakdown: {'heading': 12, 'paragraph': 37, 'list': 1, 'table': 2}


,block_id,kind,line_start,line_end,section_id,source_hash
0,B001,heading,1,1,S001,2c7133b13cb6e809
1,B002,paragraph,3,7,S001,2dece7e3e6643622
2,B003,paragraph,9,9,S001,2b348e3e1208a62a
3,B004,paragraph,11,12,S001,42ff5b9b1f329411
4,B005,heading,14,14,S002,9fa0e2af32304793
5,B006,paragraph,16,16,S002,d7e0f4e197736187
6,B007,paragraph,18,18,S002,744834dd682309f3
7,B008,heading,20,20,S003,30f1f2fd1ab8ece4
8,B009,paragraph,22,22,S003,f2fd8cba3ffc1540
9,B010,paragraph,24,24,S003,a9947391b14202d3


... 36 more rows


## 5 · Extract metadata (title, candidate authors, abstract, DOI; no hallucinated fields)

In [7]:
def extract_metadata(sections, source):
    title_candidates = [s for s in sections if s['level'] == 1 and s['kind'] != 'front_matter']
    title_section = title_candidates[0] if title_candidates else None
    title = title_section['title'] if title_section else None
    authors = []
    author_source = None
    # Deliberately conservative: explicit bold author line near the title.
    first_abstract = next((s['line_start'] for s in sections if s['kind']=='abstract'), source['line_count']+1)
    for n in range(1, min(first_abstract, 20)):
        raw = source['lines'][n-1].strip()
        match = re.fullmatch(r'\*\*(.+?)\*\*', raw)
        if not match: continue
        candidate = match.group(1).strip()
        if any(word in candidate.lower() for word in ('keyword', 'abstract', 'university', '@')): continue
        authors = [a.strip() for a in re.split(r',\s*|\s+and\s+', candidate) if a.strip()]
        author_source = n
        break
    doi_match = re.search(r'\b10\.\d{4,9}/[^\s<>"\]]+', source['text'], re.I)
    abstract_section = next((s for s in sections if s['kind']=='abstract'), None)
    abstract_text = source_lines(abstract_section['line_start']+1, abstract_section['local_end']).strip() if abstract_section and abstract_section['line_start']<abstract_section['local_end'] else None
    return {
        'title':title, 'title_source_line': title_section['line_start'] if title_section else None,
        'authors_candidate':authors,'authors_source_line':author_source,
        'doi': doi_match.group(0).rstrip('.,;') if doi_match else None,
        'abstract_raw':abstract_text
    }

metadata = extract_metadata(sections, source)
print('Title:', metadata['title'])
print('Author candidates:', metadata['authors_candidate'])
print('DOI:', metadata['doi'] or 'not found')
print('Abstract length:',len(metadata['abstract_raw'] or ''),'characters')

Title: Adaptive Test-Time Intervention for Concept Bottleneck Models
Author candidates: ['Matthew Shen\\*', 'Aliyah R. Hsu\\*', 'Abhineet Agarwal', 'and Bin Yu']
DOI: not found
Abstract length: 1162 characters


## 6 · Parse tables and attach original captions, headers, and rows

In [8]:
TABLE_CAPTION_RE = re.compile(r'^\s*(?:\*\*)?Tables?\s+([IVXLC]+|\d+)\s*[.:-]\s*(.*?)(?:\*\*)?\s*$', re.I)

def table_cells(table_markdown):
    """Use MarkdownIt table tokens, not naive `str.split('|')` (escaped pipes matter)."""
    result, row, in_cell = [], None, False
    for token in markdown_parser.parse(table_markdown):
        if token.type == 'tr_open': row = []
        elif token.type in ('th_open','td_open'): in_cell=True
        elif token.type == 'inline' and in_cell and row is not None: row.append(token.content)
        elif token.type in ('th_close','td_close'): in_cell=False
        elif token.type == 'tr_close' and row is not None: result.append(row)
    return result

def parse_tables(blocks):
    tables = []
    for block in blocks:
        if block['kind'] != 'table': continue
        prior = [b for b in blocks if b['line_end'] < block['line_start']
                 and b['section_id'] == block['section_id'] and b['kind']=='paragraph']
        prior = prior[-1] if prior else None
        caption, label, caption_line = None, None, None
        if prior and 0 <= block['line_start']-prior['line_end'] <= 4:
            ptext = source_lines(prior['line_start'],prior['line_end']).strip()
            match = TABLE_CAPTION_RE.match(ptext)
            if match:
                label = match.group(1).upper()
                caption = ptext.strip('* ')
                caption_line = prior['line_start']
        rows = table_cells(source_lines(block['line_start'],block['line_end']))
        table_id = f'TABLE_{label}' if label else f'TABLE_AUTO_{len(tables)+1:03d}'
        tables.append({
            'table_id':table_id, 'label':label,'caption':caption,'caption_line':caption_line,
            'block_id':block['block_id'],'section_id':block['section_id'],
            'line_start':block['line_start'],'line_end':block['line_end'],
            'headers':rows[0] if rows else [],'rows':rows[1:] if len(rows)>1 else [],
            'n_data_rows':max(0,len(rows)-1)
        })
    return tables

tables = parse_tables(blocks)
for tbl in tables:
    print(f"{tbl['table_id']}: {tbl['caption']}  [{tbl['line_start']}-{tbl['line_end']}], {tbl['n_data_rows']} data rows")
    show_rows([dict(zip(tbl['headers'], r)) for r in tbl['rows']], n=5)

TABLE_1: Table 1. Best CBM/TBM test prediction performance with FIGS-BD and XGBoost student models across the 4 datasets.** “Teacher Pred” and “Student Pred” denote teacher and student test prediction performance, respectively.  [73-82], 8 data rows


,Dataset,Teacher,Student,Teacher Pred,Student Pred
0,CUB (Acc %),CBM Linear,FIGS-BD,79.8,75.9
1,CUB (Acc %),–,XGBoost,–,75.9
2,TravelingBirds (Acc %),CBM Linear,FIGS-BD,51.8,47.9
3,TravelingBirds (Acc %),–,XGBoost,–,47.7
4,AGNews (Acc %),TBM Transformer,FIGS-BD,89.6,88.8


... 3 more rows
TABLE_AUTO_002: None  [102-106], 3 data rows


,Adaptive TTI strategy,Uncorrectable samples
0,FIGS,3
1,Linear,4
2,Random,4


## 7 · Find figures and available image assets; flag missing visual evidence

In [9]:
FIG_CAPTION_RE = re.compile(r'^\s*(?:\*\*)?(?:Fig(?:ure)?\.?|FIGURE)\s*(\d+)\s*[.:-]\s*(.*?)(?:\*\*)?\s*$', re.I)
IMG_MD_RE = re.compile(r'!\[[^\]]*\]\(([^\)]+)\)')
IMG_TEX_RE = re.compile(r'\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}')
IMG_HTML_RE = re.compile(r'<img\b[^>]*\bsrc\s*=\s*["\']([^"\']+)["\']', re.I)

def parse_figures(blocks, source_path):
    found_captions, image_refs = [], []
    for block in blocks:
        raw = source_lines(block['line_start'], block['line_end'])
        if block['kind']=='paragraph':
            m = FIG_CAPTION_RE.match(raw.strip())
            if m:
                found_captions.append({'label':m.group(1),'caption':raw.strip().strip('* '),
                                       'caption_line':block['line_start'], 'section_id':block['section_id']})
        for match in chain(IMG_MD_RE.finditer(raw),IMG_TEX_RE.finditer(raw),IMG_HTML_RE.finditer(raw)):
            path = match.group(1).strip().strip('<>').split(' "')[0]
            offset = raw[:match.start()].count('\n')
            image_refs.append({'path':path,'line':block['line_start']+offset,
                               'section_id':block['section_id']})
    figures = []
    claimed = set()
    for item in found_captions:
        closest = sorted(((abs(im['line']-item['caption_line']),i,im) for i, im in enumerate(image_refs)
                          if i not in claimed and im['section_id']==item['section_id']), key=lambda x:x[0])
        im = closest[0][2] if closest and closest[0][0] <= 12 else None
        if im:
            claimed.add(closest[0][1])
            if re.match(r'https?://',im['path']): status='remote_unverified'
            elif (source_path.parent/im['path']).is_file(): status='available'
            else: status='file_not_found'
        else: status='no_image_in_markdown'
        figures.append({**item,'figure_id':f"FIG_{item['label']}",
                        'asset_path':im['path'] if im else None,
                        'image_line': im['line'] if im else None, 'asset_status':status})
    unmatched = [im for i,im in enumerate(image_refs) if i not in claimed]
    return figures, unmatched

figures, uncaptioned_images = parse_figures(blocks,SOURCE_PATH)
print('Figures:'); show_rows(figures,10)
print('Image references without matched caption:',uncaptioned_images)

Figures:


,label,caption,caption_line,section_id,figure_id,asset_path,image_line,asset_status
0,1,Figure 1.** The CBM incorrectly identifies “lo...,26,S003,FIG_1,None,None,no_image_in_markdown
1,2,Figure 2.** Effectiveness of adaptive test-tim...,49,S007,FIG_2,None,None,no_image_in_markdown
2,3,Figure 3.** Performance of CBM linear with ada...,86,S010,FIG_3,None,None,no_image_in_markdown
3,4,Figure 4.** Left: number of uncorrectable samp...,98,S010,FIG_4,None,None,no_image_in_markdown


Image references without matched caption: []


## 8 · Capture display equations, tags, and line-based evidence locations

In [10]:
EQUATION_PATTERNS = [
    ('dollars',re.compile(r'(?m)^[ \t]*\$\$[ \t]*\r?\n(.*?)[ \t]*\r?\n[ \t]*\$\$[ \t]*(?=\r?$)', re.S|re.M)),
    ('brackets',re.compile(r'\\\[(.*?)\\\]', re.S)),
    ('environment',re.compile(r'\\begin\{(?:equation\*?|align\*?)\}(.*?)\\end\{(?:equation\*?|align\*?)\}', re.S)),
]

def parse_equations(source, blocks):
    code_ranges = [(b['line_start'],b['line_end']) for b in blocks if b['kind']=='code']
    equations, already = [], []
    for delimiter,pattern in EQUATION_PATTERNS:
        for m in pattern.finditer(source['text']):
            if any(a <= m.start() < b or m.start() <= a < m.end() for a,b in already): continue
            start = char_to_line(m.start())
            end = char_to_line(max(m.start(),m.end()-1))
            if any(not (end < a or start > b) for a,b in code_ranges): continue
            tag = re.search(r'\\tag\{([^}]+)\}',m.group(0))
            equations.append({
                'equation_id':f'EQ_{tag.group(1)}' if tag else f'EQ_AUTO_{len(equations)+1:03d}',
                'tag':tag.group(1) if tag else None, 'delimiter':delimiter,
                'line_start':start,'line_end':end,
                'section_id':section_for_line(start)['section_id'] if section_for_line(start) else None,
                'latex':m.group(1).strip(), 'source_hash':sha256(m.group(0).encode()).hexdigest()[:16]
            })
            already.append((m.start(),m.end()))
    return sorted(equations,key=lambda eq:eq['line_start'])

equations = parse_equations(source,blocks)
show_rows([{k:v for k,v in eq.items() if k!='latex'} for eq in equations], n=15)
print('Example equation:\n', equations[0]['latex'] if equations else '(none)')

,equation_id,tag,delimiter,line_start,line_end,section_id,source_hash
0,EQ_AUTO_001,None,dollars,55,57,S007,f88346614861fbf6


Example equation:
 f:\{0,1\}^{d}\rightarrow\mathbb{R}.


## 9 · Parse numbered bibliography entries into separate retrievable records

In [11]:
BIB_ENTRY_RE = re.compile(r'^\s*\[(\d+)\]\s+(.*)')
YEAR_RE = re.compile(r'\b(?:19\d{2}|20\d{2})\b')

def parse_bibliography(sections):
    entries = []
    bib_sections = [s for s in sections if s['kind']=='bibliography']
    for section in bib_sections:
        active = None
        for line_no in range(section['line_start']+1,section['subtree_end']+1):
            line = source['lines'][line_no-1].rstrip('\r\n')
            m = BIB_ENTRY_RE.match(line)
            if m:
                if active: entries.append(active)
                active = {'reference_id':f'REF{int(m.group(1)):03d}',
                          'marker':f'[{m.group(1)}]', 'number':int(m.group(1)),
                          'line_start':line_no,'line_end':line_no,'raw':m.group(2).strip(),
                          'section_id':section['section_id']}
            elif active and line.strip():
                active['raw'] += ' ' + line.strip()
                active['line_end']=line_no
        if active: entries.append(active)
    for entry in entries:
        years = YEAR_RE.findall(entry['raw'])
        entry['year_candidate'] = int(years[-1]) if years else None
    return entries

bibliography = parse_bibliography(sections)
print('Bibliography entries:',len(bibliography))
show_rows([{k:v for k,v in r.items() if k!='raw'} for r in bibliography], n=7)
print('REF002:', next((r['raw'][:200] for r in bibliography if r['number']==2),None))

Bibliography entries: 0


""


REF002: None


## 10 · Locate in-text citation occurrences and map to bibliography IDs

In [12]:
BRACKET_CITATION_RE = re.compile(r'\[((?:\d+\s*[,;]\s*)*\d+)\]')
BRACKET_RANGE_RE = re.compile(r'\[(\d+)\]\s*[-–—]\s*\[(\d+)\]')
LATEX_CITATION_RE = re.compile(r'\\cite(?:\[[^\]]*\])?\{([^}]+)\}')

def parse_citations(blocks, bibliography):
    present_nums = {r['number'] for r in bibliography}
    hits = []
    for block in blocks:
        if block['kind'] in ('code','heading','table'):continue
        section = section_by_id.get(block['section_id'])
        if section and section['kind']=='bibliography':continue
        raw = source_lines(block['line_start'],block['line_end'])
        # Expand citations such as [4]–[8] into 4, 5, 6, 7, 8 (not just endpoints).
        occupied=[]
        for m in BRACKET_RANGE_RE.finditer(raw):
            lo,hi=int(m.group(1)),int(m.group(2))
            if lo>hi or hi-lo>100: continue  # suspicious span: leave for manual inspection
            occupied.append(m.span())
            line=block['line_start']+raw[:m.start()].count('\n')
            for number in range(lo,hi+1):
                hits.append({'reference_id':f'REF{number:03d}', 'marker':m.group(0),
                             'line':line,'section_id':block['section_id'],
                             'block_id':block['block_id'], 'resolved':number in present_nums})
        for m in BRACKET_CITATION_RE.finditer(raw):
            if any(m.start()>=a and m.end()<=b for a,b in occupied): continue
            numbers=[int(k.strip()) for k in re.split(r'[,;]',m.group(1))]
            line=block['line_start']+raw[:m.start()].count('\n')
            for number in numbers:
                hits.append({'reference_id':f'REF{number:03d}', 'marker':m.group(0),
                             'line':line,'section_id':block['section_id'],
                             'block_id':block['block_id'], 'resolved':number in present_nums})
        for m in LATEX_CITATION_RE.finditer(raw):
            line=block['line_start']+raw[:m.start()].count('\n')
            for key in m.group(1).split(','):
                hits.append({'reference_id':key.strip(),'marker':m.group(0),
                             'line':line,'section_id':block['section_id'],
                             'block_id':block['block_id'], 'resolved':False})
    return hits

citations=parse_citations(blocks,bibliography)
print('In-text citation occurrences:',len(citations))
print('Distinct numbered references:',len({c['reference_id'] for c in citations if c['resolved']}))
show_rows(citations, 14)

In-text citation occurrences: 0
Distinct numbered references: 0


""


## 11 · Resolve Table / Fig / Equation mentions to scientific objects

In [13]:
CROSSREF_RE = re.compile(r'\b(?P<type>Tables?|Figures?|Figs?\.?|Equations?|Eqs?\.?)\s+(?P<label>[IVXLC]+|\d+)\b',re.I)

def parse_crossrefs(blocks,tables,figures,equations):
    lookup={}
    lookup.update({('table',t['label']):t['table_id'] for t in tables if t['label']})
    lookup.update({('figure',f['label']):f['figure_id'] for f in figures})
    lookup.update({('equation',e['tag']):e['equation_id'] for e in equations if e['tag']})
    out=[]
    for block in blocks:
        if block['kind'] not in ('paragraph','list','blockquote'):continue
        section=section_by_id.get(block['section_id'])
        if section and section['kind']=='bibliography':continue
        raw=source_lines(block['line_start'],block['line_end'])
        for m in CROSSREF_RE.finditer(raw):
            line=block['line_start']+raw[:m.start()].count('\n')
            # Caption labels are definitions, not references.
            current_line=source['lines'][line-1].strip().lstrip('*').strip()
            if re.match(r'^(?:Table|Fig(?:ure)?\.?)\s+[IVXLC\d]+\s*\.',current_line,re.I):continue
            word=m.group('type').lower().rstrip('.')
            kind=('table' if word.startswith('table') else 'figure' if word.startswith(('fig','figure')) else 'equation')
            label=m.group('label').upper() if kind=='table' else m.group('label')
            obj_id=lookup.get((kind,label))
            out.append({'kind':kind,'label':label,'mention':m.group(0),
                        'line':line,'section_id':block['section_id'],
                        'object_id':obj_id,'resolved':bool(obj_id)})
    return out

crossrefs=parse_crossrefs(blocks,tables,figures,equations)
print('Cross-references:',len(crossrefs))
show_rows(crossrefs,20)
print('Unresolved:',[f"{x['mention']} @ L{x['line']}" for x in crossrefs if not x['resolved']])

Cross-references: 5


,kind,label,mention,line,section_id,object_id,resolved
0,figure,1,Figure 1,24,S003,FIG_1,True
1,table,1,Table 1,69,S009,TABLE_1,True
2,figure,3,Figure 3,94,S010,FIG_3,True
3,figure,2,Figure 2,96,S010,FIG_2,True
4,figure,4,Figure 4,112,S010,FIG_4,True


Unresolved: []


## 12 · Create bounded reading chunks (no splitting inside a table or math block)

In [14]:
def estimate_tokens(text):
    # Safe, reproducible approximation; replace with your model's tokenizer later.
    return max(1, len(re.findall(r'\w+|[^\w\s]',text, flags=re.UNICODE)))

def create_chunks(sections,blocks,max_tokens=MAX_CHUNK_TOKENS):
    by_section=defaultdict(list)
    for block in blocks:
        if block['kind']!='heading': by_section[block['section_id']].append(block)
    chunks=[]
    for section in sections:
        group=by_section.get(section['section_id'],[])
        if not group: continue  # heading-only parents have no independent reading unit
        current=[]; current_tokens=0
        def emit():
            nonlocal current,current_tokens
            if not current: return
            start,end=current[0]['line_start'],current[-1]['line_end']
            cid=f'CH{len(chunks)+1:03d}'
            chunks.append({'chunk_id':cid,'section_id':section['section_id'],
                           'heading_path':section['heading_path'], 'line_start':start,
                           'line_end':end,'block_ids':[b['block_id'] for b in current],
                           'estimated_tokens':estimate_tokens(source_lines(start,end)),
                           'mandatory_reading':section['kind'] not in ('bibliography','back_matter'),
                           'oversized':estimate_tokens(source_lines(start,end))>max_tokens,
                           'source_hash':sha256(source_lines(start,end).encode()).hexdigest()[:16]})
            current=[];current_tokens=0
        for block in group:
            raw=source_lines(block['line_start'],block['line_end'])
            tokens_needed=estimate_tokens(raw)
            if current and current_tokens+tokens_needed>max_tokens:emit()
            current.append(block);current_tokens+=tokens_needed
            if tokens_needed>max_tokens:emit()  # oversized atomic block flagged, never silently cut
        emit()
    return sorted(chunks,key=lambda c:c['line_start'])

chunks=create_chunks(sections,blocks)
chunks_by_id={c['chunk_id']:c for c in chunks}
show_rows([{k:v for k,v in c.items() if k not in ('block_ids','source_hash','heading_path')}
           for c in chunks], n=25)
print('Mandatory reading chunks:',sum(c['mandatory_reading'] for c in chunks),
      '| Oversized atomic units:',sum(c['oversized'] for c in chunks))

,chunk_id,section_id,line_start,line_end,estimated_tokens,mandatory_reading,oversized
0,CH001,S001,3,12,115,True,False
1,CH002,S002,16,18,212,True,False
2,CH003,S003,22,32,630,True,False
3,CH004,S005,37,37,186,True,False
4,CH005,S006,41,41,86,True,False
5,CH006,S007,45,59,430,True,False
6,CH007,S008,63,65,258,True,False
7,CH008,S009,69,82,491,True,False
8,CH009,S010,86,112,941,True,False
9,CH010,S010,114,116,215,True,False


Mandatory reading chunks: 12 | Oversized atomic units: 0


## 13 · Source retrieval tools: `read_section`, `read_chunk`, `read_table`, `search_paper`

In [15]:
def read_section(section_id, include_children=False, with_line_numbers=True):
    section=section_by_id[section_id]
    last=section['subtree_end'] if include_children else section['local_end']
    rows=[f'L{n:04d} | {source["lines"][n-1].rstrip()}' for n in range(section['line_start'],last+1)]
    return '\n'.join(rows) if with_line_numbers else source_lines(section['line_start'],last)

def read_chunk(chunk_id, heading_context=True, with_line_numbers=True):
    chunk=chunks_by_id[chunk_id]
    section=section_by_id[chunk['section_id']]
    prefix=' > '.join(section['heading_path'])+'\n' if heading_context else ''
    rows=[f'L{n:04d} | {source["lines"][n-1].rstrip()}' for n in range(chunk['line_start'],chunk['line_end']+1)]
    return prefix+('\n'.join(rows) if with_line_numbers else source_lines(chunk['line_start'],chunk['line_end']))

def read_table(table_id):
    table=next(t for t in tables if t['table_id']==table_id)
    begin=table['caption_line'] or table['line_start']
    return source_lines(begin,table['line_end'])

def search_paper(query, top_k=8, section_ids=None, include_bibliography=False):
    """Simple lexical local retrieval. Returns raw source snippets, never generated summaries."""
    words=[w.lower() for w in re.findall(r'\w+',query) if len(w)>1]
    if not words:return []
    scored=[]
    for block in blocks:
        if block['kind']=='heading':continue
        sec=section_by_id.get(block['section_id'])
        if section_ids is not None and block['section_id'] not in section_ids:continue
        if not include_bibliography and sec and sec['kind']=='bibliography':continue
        raw=source_lines(block['line_start'],block['line_end'])
        lower=raw.lower()
        score=sum(lower.count(word) for word in words)+4*lower.count(query.lower())
        if score:
            scored.append({'score':score,'block_id':block['block_id'],
                           'section_id':block['section_id'],
                           'line_start':block['line_start'],'line_end':block['line_end'],
                           'snippet':raw.strip()[:500]})
    return sorted(scored,key=lambda x:(-x['score'],x['line_start']))[:top_k]

# Run these interactive debugging examples or change the arguments.
if any(t['table_id']=='TABLE_II' for t in tables):
    print('Table II raw:'); print(read_table('TABLE_II')[:700])
else:
    print('No Table II in this manuscript; skipping example.')
print('\nSearch: "OOD AUROC":')
show_rows(search_paper('OOD AUROC',top_k=5),n=5)

No Table II in this manuscript; skipping example.

Search: "OOD AUROC":


,score,block_id,section_id,line_start,line_end,snippet
0,1,B047,S010,114,114,"As a case study, we highlight an example from ..."


## 14 · Validate source coverage, provenance, hierarchy, assets, and links

In [16]:
def validate_preprocessing():
    errors,warnings,checks=[],[],{}
    line_count=source['line_count']
    # Each original line must have exactly one local section owner.
    owners=Counter()
    for sec in sections:
        for line in range(sec['line_start'],sec['local_end']+1):owners[line]+=1
    owner_bad=[n for n in range(1,line_count+1) if owners[n]!=1]
    if owner_bad:errors.append(f'{len(owner_bad)} source lines have !=1 section owner: {owner_bad[:10]}')
    checks['all_lines_owned_once']=not bool(owner_bad)
    # Block coverage of non-blank source lines. Original line mapping is immutable.
    coverage=set(chain.from_iterable(range(b['line_start'],b['line_end']+1) for b in blocks))
    missed=[n for n,line in enumerate(source['lines'],1) if line.strip() and n not in coverage]
    if missed:errors.append(f'{len(missed)} nonblank lines missed by top-level blocks: {missed[:20]}')
    checks['all_nonblank_lines_in_blocks']=not bool(missed)
    # Source hash and offsets must resolve.
    for block in blocks:
        start,end=block['line_start'],block['line_end']
        if not (1<=start<=end<=line_count):errors.append('invalid block range '+block['block_id']);continue
        computed=sha256(source_lines(start,end).encode('utf-8')).hexdigest()[:16]
        if computed!=block['source_hash']:errors.append('block hash mismatch '+block['block_id'])
    checks['source_spans_integrity']=not any('block ' in e for e in errors)
    # A reading chunk must reference blocks in the same local section.
    for chunk in chunks:
        if chunk['line_start']>chunk['line_end']:errors.append('invalid chunk '+chunk['chunk_id'])
        for bid in chunk['block_ids']:
            if blocks_by_id[bid]['section_id']!=chunk['section_id']:
                errors.append('chunk mixed sections '+chunk['chunk_id'])
        if chunk['oversized']:warnings.append('Oversized indivisible chunk '+chunk['chunk_id'])
    for fig in figures:
        if fig['asset_status']!='available':warnings.append(f"{fig['figure_id']}: {fig['asset_status']}")
    for im in uncaptioned_images:warnings.append(f"Uncaptioned image link at L{im['line']}: {im['path']}")
    for r in crossrefs:
        if not r['resolved']:warnings.append(f"Unresolved {r['mention']} at L{r['line']}")
    for c in citations:
        if not c['resolved']:warnings.append(f"Unresolved citation {c['marker']} at L{c['line']}")
    for tbl in tables:
        if not tbl['caption']:warnings.append(f"{tbl['table_id']}: caption not located")
        if len(set(map(len,tbl['rows'])))>1 or any(len(row)!=len(tbl['headers']) for row in tbl['rows']):
            warnings.append(f"{tbl['table_id']}: inconsistent row widths")
    checks['no_errors']=not errors
    return {'checks':checks,'errors':errors,'warnings':sorted(set(warnings)),
            'counts':{'sections':len(sections),'blocks':len(blocks),'chunks':len(chunks),
                      'tables':len(tables),'figures':len(figures),'equations':len(equations),
                      'bibliography':len(bibliography),'citation_mentions':len(citations)}}

validation=validate_preprocessing()
print('Checks:'); pprint(validation['checks'])
print('Counts:'); pprint(validation['counts'])
print('\nERRORS:',validation['errors'])
print('\nWARNINGS (do not silently fix original evidence):')
for warn in validation['warnings']:print('  -',warn)
assert not validation['errors'], 'Preprocessing has structural errors; inspect the report.'

Checks:
{'all_lines_owned_once': True,
 'all_nonblank_lines_in_blocks': True,
 'no_errors': True,
 'source_spans_integrity': True}
Counts:
{'bibliography': 0,
 'blocks': 52,
 'chunks': 12,
 'citation_mentions': 0,
 'equations': 1,
 'figures': 4,
 'sections': 12,
 'tables': 2}

ERRORS: []

WARNINGS (do not silently fix original evidence):
  - FIG_1: no_image_in_markdown
  - FIG_2: no_image_in_markdown
  - FIG_3: no_image_in_markdown
  - FIG_4: no_image_in_markdown
  - TABLE_AUTO_002: caption not located


## 15 · Build and export canonical parsed objects (JSON, no source rewrite)

In [17]:
def assemble_parsed_paper():
    return {
        'schema_version':PARSER_VERSION,
        'source':{'path':source['path'],'sha256':source['sha256'],
                  'line_count':source['line_count'],'character_count':len(source['text'])},
        'metadata':metadata,'sections':sections,'blocks':blocks,'tables':tables,
        'figures':figures,'unmatched_image_refs':uncaptioned_images,
        'equations':equations,'bibliography':bibliography,'citations':citations,
        'crossrefs':crossrefs,'chunks':chunks,'validation':validation
    }

parsed_paper=assemble_parsed_paper()
section_catalog=[{
    'section_id':s['section_id'],'title':s['title'],
    'heading_path':s['heading_path'],'parent_id':s['parent_id'],
    'kind':s['kind'],'line_start':s['line_start'],
    'local_end':s['local_end'],'subtree_end':s['subtree_end'],
    'chunk_ids':[c['chunk_id'] for c in chunks if c['section_id']==s['section_id']]
} for s in sections]
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
parsed_path=OUTPUT_DIR/'ParsedPaper.json'
catalog_path=OUTPUT_DIR/'SectionCatalog.json'
parsed_path.write_text(json.dumps(parsed_paper,ensure_ascii=False,indent=2),encoding='utf-8')
catalog_path.write_text(json.dumps(section_catalog,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:',parsed_path)
print('Saved:',catalog_path)
print('Original input SHA-256:',source['sha256'])

Saved: paper_parser_outputs/ParsedPaper.json
Saved: paper_parser_outputs/SectionCatalog.json
Original input SHA-256: b980864d727aac686fbce1f59fa6f96cade769cab3290b34f1ab8ddcfdc04ed6


## 16 · Assertions for **this specific uploaded paper** (regression checks)

In [18]:
if RUN_EDLECOC_REGRESSION:
    # These are known features of the supplied EDLECOC Markdown, NOT inferred generic guarantees.
    assert len(tables)==2, f'Expected 2 populated tables, found {len(tables)}'
    assert {t['label'] for t in tables}=={'II','III'}
    assert len(bibliography)==20, f'Expected 20 refs, found {len(bibliography)}'
    assert 'TUBITAK' not in bibliography[-1]['raw'], 'Acknowledgements must not contaminate REF020'
    assert bibliography[-1]['line_end']==191, 'Final reference should end on the actual reference line'
    assert any(s['kind']=='back_matter' for s in sections), 'Acknowledgements footer should be separate from bibliography'
    assert len(equations)==5, f'Expected five display equations, found {len(equations)}'
    assert [eq['tag'] for eq in equations]==['1','2','3','4','5']
    assert any(f['figure_id']=='FIG_1' and f['asset_status']=='no_image_in_markdown' for f in figures)
    assert any(x['kind']=='table' and x['label']=='I' and not x['resolved'] for x in crossrefs), 'Table I missing from OCR Markdown should be flagged'
    assert any(x['kind']=='table' and x['label']=='II' and x['resolved'] for x in crossrefs)
    assert any(c['reference_id']=='REF020' for c in citations), 'Citation [20] must be mapped'
    assert any(c['reference_id']=='REF006' and c['marker']=='[4]–[8]' for c in citations), 'Citations with ranges must expand correctly'
    assert any(c['reference_id']=='REF014' and c['marker']=='[13]–[15]' for c in citations), 'Citation range 13–15 must include 14'
    assert any('EDL-ECOC' in ' '.join(t['headers']+list(chain.from_iterable(t['rows']))) for t in tables)
    assert any('0.619 ± 0.013' in str(x) for tbl in tables for row in tbl['rows'] for x in row)
    assert any('0.696 ± 0.022' in str(x) for tbl in tables for row in tbl['rows'] for x in row)
    assert not validation['errors']
    assert sha256(SOURCE_PATH.read_bytes()).hexdigest()==source['sha256'], 'Source changed during notebook run'
    print('PASS — All example-specific regression checks succeeded.')
    print('Expected source limitations correctly surfaced: Table I absent, Fig. 1 image absent.')
else:
    print("Skipped example-specific EDLECOC debug cell.")

Skipped example-specific EDLECOC debug cell.
